# sisl 01 · From SIESTA matrices to bands and density

[Open in Colab](https://colab.research.google.com/github/YHKlab-RGLee/tutorial/blob/main/sisl/01/Sisl_Matrices.ipynb)

Run the cells in order using a **CPU runtime**. This tutorial introduces `Hamiltonian`, `Overlap`, `DensityMatrix`, and `Grid` with a four-atom bulk BN example. We call sisl's public API directly, without implementing custom calculation functions or a tight-binding model.

Workflow: SIESTA SCF → read and visualize matrices → diagonalize → build a supercell and examine band folding → evaluate density on several grids → DOS and wavefunctions.

Only the structure and B/N PSF files come from the supplied BN example. The basis, k mesh, and cutoff are chosen for a short exercise; they are not converged research settings.

## 1. Install SIESTA from conda-forge

Use micromamba to install the **SIESTA 5.4.2 serial** executable. Python cells use Colab's Python and **sisl 0.16.4**, installed with pip. No compilation, kernel replacement, or restart is required.

This follows the official [SIESTA conda installation guide](https://docs.siesta-project.org/projects/siesta/en/stable/installation/conda.html). Installation may take a few minutes.

In [ ]:
%%bash
set -e
mkdir -p /content/mamba
curl -Ls --fail https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content/mamba bin/micromamba
/content/mamba/bin/micromamba create -y -r /content/mamba -p /content/siesta-env     -c conda-forge --override-channels 'siesta=5.4.2=nompi_*'
/content/siesta-env/bin/siesta --version

In [ ]:
%pip install -q sisl==0.16.4 matplotlib netCDF4
import os
import numpy as np
import matplotlib.pyplot as plt
import sisl as si
from pathlib import Path

os.environ["OMP_NUM_THREADS"] = "2"
print("sisl:", si.__version__)

## 2. Prepare the inputs and run SIESTA

Download the four input files from this repository. The Colab link becomes usable once this tutorial is pushed to the repository. To use another branch, replace `main` in `base` below.

In [ ]:
from urllib.request import urlretrieve

work = Path("/content/bn-matrices")
work.mkdir(exist_ok=True)
base = "https://raw.githubusercontent.com/YHKlab-RGLee/tutorial/main/sisl/01/src/inputs"
for name in ["RUN.fdf", "STRUCT.fdf", "B.psf", "N.psf"]:
    urlretrieve(f"{base}/{name}", work / name)
os.chdir(work)
print(Path("STRUCT.fdf").read_text())

### Which files does each object need?

| Task / sisl object | Required SIESTA files | Output option or input |
|---|---|---|
| SCF calculation | `RUN.fdf`, `STRUCT.fdf`, `B.psf`, `N.psf` | Structure, pseudopotentials, SZP basis, LDA/CA |
| `Hamiltonian`, `Overlap` / bands, DOS | `BN.HSX` and geometry information | `SaveHS true`; read through `RUN.fdf` |
| Energy reference $E_F$ | `BN.HSX` | Fermi level stored in HSX, in eV |
| `DensityMatrix` | `BN.DM` and geometry/basis information | `Write.DM true` |
| Real-space density, wavefunction | The matrices above and **`B.ion.xml`, `N.ion.xml`** | `WriteIonXML true`: numerical atomic orbitals |
| Comparison with SIESTA density | `BN.RHO` | `SaveRho true` |

A `PSF` contains a pseudopotential. `DM.density()` also needs real-space orbital functions, so PSF files alone are insufficient. Keep the `ion.xml` files in the calculation directory. HSX provides the matrices needed for bands; reconstructing density also requires the DM and numerical orbitals.

Read the complete input below. `MD.NumCGsteps 0` selects a fixed-geometry SCF calculation, and `DM.UseSaveDM false` disables restarting from a previous DM. The installation and execution commands are shown directly.

In [ ]:
print(Path("RUN.fdf").read_text())

In [ ]:
%%bash
set -e
OMP_NUM_THREADS=2 OPENBLAS_NUM_THREADS=1 /content/siesta-env/bin/siesta < RUN.fdf > siesta.out
grep -E 'SCF Convergence|SCF_NOT_CONV|Fermi|Total =' siesta.out | tail -12

In [ ]:
required = ["BN.HSX", "BN.DM", "BN.RHO", "B.ion.xml", "N.ion.xml"]
assert all(Path(name).is_file() for name in required)
log = Path("siesta.out").read_text()
assert "SCF Convergence" in log and "SCF_NOT_CONV" not in log
print("SCF convergence and required output files verified")

## 3. Read the matrix objects

`get_sile()` selects a reader for the file format. The FDF reader connects geometry, basis, and matrix files. The `order` argument explicitly selects the matrix file format used in this exercise.

SIESTA's PAO basis is **nonorthogonal**, so we solve $H(k)c=E S(k)c$. `H` also stores the overlap, and we can read a separate `Overlap` object, `S`.

In [ ]:
fdf = si.get_sile("RUN.fdf")
H = fdf.read_hamiltonian(order=["HSX"])
S = fdf.read_overlap(order=["HSX"])
DM = fdf.read_density_matrix(order=["DM"])
Ef = si.get_sile("BN.HSX").read_fermi_level()
print(H)
print(S)
print(DM)
print("Fermi level [eV]:", Ef)

### Atom indices, orbital indices, and neighboring cells

`na` counts atoms in the reference cell, and `no` counts its orbitals. `a2o()` maps atom indices to orbital indices. Before reading a matrix shape, distinguish two concepts:

- **Basis overlap:** different PAOs are not generally orthogonal, so their overlaps form a matrix $S$.
- **Neighbor-cell storage:** an orbital in the reference cell can couple to an orbital in another periodic image. These connections need a cell index as well as an orbital index.

Let $\phi_{i,\mathbf 0}$ be orbital $i$ in the reference cell and $\phi_{j,\mathbf R}$ be orbital $j$ in the cell translated by $\mathbf R$. Then

$$
H_{ij}(\mathbf R)=\langle\phi_{i,\mathbf 0}|\hat H|\phi_{j,\mathbf R}\rangle,
\qquad
S_{ij}(\mathbf R)=\langle\phi_{i,\mathbf 0}|\phi_{j,\mathbf R}\rangle.
$$

Each $H(\mathbf R)$ or $S(\mathbf R)$ is an $N_{\mathrm{orb}}\times N_{\mathrm{orb}}$ block. sisl stores the blocks side by side:

$$
H_{\mathrm{stored}}=
\underbrace{\begin{bmatrix}
H(\mathbf 0)&H(\mathbf R_1)&H(\mathbf R_2)&\cdots
\end{bmatrix}}_{N_{\mathrm{orb}}\ \mathrm{rows}\;\times\;N_{\mathrm{orb}}N_{\mathrm{cells}}\ \mathrm{columns}}.
$$

**Every row belongs to the reference cell. Each column identifies both an orbital and a cell.** This is why the stored matrix is rectangular. `geometry.lattice.sc_off` lists the integer cell offsets in storage order; do not assume the blocks follow increasing distance.

For this BN calculation, `no = 36` and `nsc = [7, 7, 3]`. There are $7\times7\times3=147$ cell-offset slots, giving $36\times(36\times147)=36\times5292$ spatial entries before sparse storage. **The reference geometry still has only four atoms and 36 orbitals.** The 147 slots do not represent 147 independently treated cells, and many matrix entries are zero. `nsc` describes the available range of connections; it does not imply every pair interacts.

`DensityMatrix` uses the same reference-cell/neighbor-cell indexing for $D_{i\mathbf0,j\mathbf R}$. Its entries describe the electronic density in the orbital basis, rather than hopping amplitudes. The DM file contains a sparse representation; this does not imply that the exact density matrix must vanish wherever the overlap vanishes.

The last axis of `H.shape` or `DM.shape` contains matrix components. In this nonmagnetic, nonorthogonal example, its size is 2: the H or DM component and the overlap component. It is **not** an additional cell or two spin channels.

In [ ]:
print("atoms / orbitals:", H.geometry.na, H.no)
print("real-space shape:", H.shape, "neighbor cells:", H.geometry.nsc)
print("atom 0 orbitals:", H.geometry.a2o(0, all=True))
print("onsite H00 [eV]:", H[0, 0, 0])
print("onsite S00:", S[0, 0])
print("orbital cutoff [Å]:", DM.geometry.maxR())
assert DM.geometry.maxR() > 0

## 4. Visualize the matrices

First inspect the sparsity pattern of the stored real-space matrices. The horizontal axis runs through **orbital blocks from neighboring cells**, while the vertical axis covers only reference-cell orbitals. Each group of `H.no` columns corresponds to one cell-offset block. `spy()` shows the locations of stored entries.

Keep this rectangular storage layout in mind when comparing it with the square Bloch matrices below.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
for ax, matrix, title in zip(axes, [H, S], ["H(R)", "S(R)"]):
    ax.spy(matrix.tocsr(), markersize=1, aspect="auto")
    ax.set(title=title, xlabel="orbital + neighboring cell", ylabel="orbital")
plt.tight_layout()

### From rectangular storage to a square Bloch matrix

At a chosen wavevector, Bloch phases combine the neighbor-cell blocks:

$$
H(\mathbf k)=\sum_{\mathbf R}e^{i\mathbf k\cdot\mathbf R}H(\mathbf R),
\qquad
S(\mathbf k)=\sum_{\mathbf R}e^{i\mathbf k\cdot\mathbf R}S(\mathbf R).
$$

These formulas use sisl's default lattice-vector gauge. If the k coordinates supplied to sisl are fractional reciprocal coordinates $\mathbf q$ and the integer cell offset is $\mathbf n$, then $\mathbf k\cdot\mathbf R=2\pi\mathbf q\cdot\mathbf n$.

$$
\underbrace{[\ H(\mathbf0)\;H(\mathbf R_1)\;\cdots\ ]}_{36\times5292}
\quad\xrightarrow{\text{Bloch sum at one }\mathbf k}\quad
\underbrace{H(\mathbf k)}_{36\times36}.
$$

**At Γ, every phase is 1, but all neighbor-cell blocks still contribute:**

$$H(\Gamma)=\sum_{\mathbf R}H(\mathbf R)\ne H(\mathbf0)\quad\text{in general}.$$

Thus a Γ-point calculation still includes periodic connections. It does not isolate the reference cell. The square matrix describes the same 36 basis orbitals at one Bloch wavevector; no orbitals have been removed. `H.Hk()` and `H.Sk()` perform these sums directly.

In [ ]:
k = [0.0, 0.0, 0.0]
Hk = H.Hk(k=k, format="array")
Sk = H.Sk(k=k, format="array")
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, matrix, title in zip(axes, [Hk, Sk], ["|H(Γ)| [eV]", "|S(Γ)|"]):
    im = ax.imshow(np.abs(matrix), origin="lower")
    ax.set(title=title, xlabel="orbital j", ylabel="orbital i")
    fig.colorbar(im, ax=ax)
plt.tight_layout()

In [ ]:
np.testing.assert_allclose(Hk, Hk.conj().T, atol=1e-8)
np.testing.assert_allclose(Sk, S.Sk(k=k, format="array"), atol=1e-8)
print("Smallest eigenvalue of S(k):", np.linalg.eigvalsh(Sk).min())
print("Γ eigenvalues [eV relative to EF]:", H.eigh(k=k) - Ef)

## 5. Diagonalize the saved Hamiltonian to calculate bands

**No additional SIESTA calculation or `.bands` file is needed.** `H.eigh()` solves the generalized eigenvalue problem including overlap. Calling only `np.linalg.eigh(Hk)` would solve a different problem.

`BandStructure` takes fractional reciprocal-lattice coordinates. For this lattice with a 120° in-plane angle, we choose Γ–M–K–Γ. `apply.array.eigh()` applies `H.eigh()` at each k point along the path. All energies are referenced to $E_F$ from the same SCF calculation.

In [ ]:
points = np.array([[0, 0, 0], [0.5, 0, 0], [1/3, 1/3, 0], [0, 0, 0]])
names = ["Γ", "M", "K", "Γ"]
bs = si.BandStructure(H, points, 121, names=names)
bands = bs.apply.array.eigh() - Ef
x = bs.lineark()
ticks, labels = bs.lineartick()
print("k points × bands:", bands.shape)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(x, bands, color="tab:blue", lw=0.8)
plt.xticks(ticks, labels)
plt.axhline(0, color="gray", ls=":")
plt.ylabel("E − EF [eV]")
plt.ylim(-15, 15)
plt.xlim(x[0], x[-1])
plt.grid(axis="x")

## 6. Expand the Hamiltonian and examine band folding

Here, expansion means **repeating the cell**. `H.tile(2, axis=0)` replicates the atoms, orbitals, hopping, and overlap along the first lattice vector. This preserves the physical crystal without applying strain or repeating the SCF calculation. Doubling the number of orbitals per cell doubles the number of bands at each k point.

### Reorganizing the reference cell, not extending `nsc`

`tile()` makes a larger reference cell and remaps existing connections into its internal and neighboring-cell blocks. A connection that crossed an old cell boundary may now lie inside the new cell. Connections across the new boundary are retained. Both Hamiltonian and overlap follow this remapping; simply copying diagonal blocks would lose the connections between the repeated parts.

| Operation | Reference-cell orbitals | What happens to neighbor connections? |
|---|---|---|
| Read the original `H` / `DM` | 36 | Stored with explicit cell offsets |
| Form `H.Hk()` / `H.Sk()` | 36 | Bloch-summed into a 36 × 36 matrix |
| Apply `H.tile(2, axis=0)` | 72 | Remapped to a larger reference cell, still with explicit cell offsets |

The same repetition is available as `DM.tile(2, axis=0)`. It repeats the existing periodic density matrix; it does not calculate a new self-consistent density for a strained or defective structure. Here we tile **H for the band comparison**, while the density exercises below continue to use the **original four-atom DM**.

### Compare along the same physical k path

For the supercell, $a'_1=2a_1$ and $b'_1=b_1/2$. The original coordinates $(q_1,q_2,q_3)$ therefore correspond to supercell coordinates $(2q_1,q_2,q_3)$ at the **same Cartesian k**. Using identical numerical coordinates for both objects would trace different physical paths.

The plot below uses an extended-zone representation along the original physical Γ–M–K–Γ path. These labels do not denote the supercell's M/K points. The shorter reciprocal period introduces additional folded branches.

In [ ]:
H2 = H.tile(2, axis=0)
bs2 = si.BandStructure(H2, points * [2, 1, 1], 121, names=names)
bands2 = bs2.apply.array.eigh() - Ef
np.testing.assert_allclose(bs.k @ H.geometry.rcell, bs2.k @ H2.geometry.rcell)
print("orbitals:", H.no, "→", H2.no)
print("bands:", bands.shape, "→", bands2.shape)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(x, bands2, color="tab:orange", lw=1, alpha=0.7)
plt.plot(x, bands, color="black", lw=0.6, ls="--")
plt.xticks(ticks, labels)
plt.ylabel("E − EF [eV]")
plt.title("2× cell (orange), original cell (black): same physical k")
plt.ylim(-15, 15)
plt.xlim(x[0], x[-1])
plt.grid(axis="x")

### See folding in the smaller Brillouin zone

The path $q_1=-1/4\ldots1/4$, with the other coordinates zero, connects the supercell zone boundaries. Combining the original-cell eigenvalues $E(q)$ and $E(q+(1/2,0,0))$ gives all supercell eigenvalues. The comparison below checks folding numerically as well as visually.

In [ ]:
fold = si.BandStructure(H, [[-0.25, 0, 0], [0.25, 0, 0]], 81)
fold2 = si.BandStructure(H2, [[-0.5, 0, 0], [0.5, 0, 0]], 81)
e0 = fold.apply.array.eigh()
e1 = np.array([H.eigh(k=q + [0.5, 0, 0]) for q in fold.k])
e2 = fold2.apply.array.eigh()
expected = np.sort(np.concatenate([e0, e1], axis=1), axis=1)
print("folding error [eV]:", np.max(np.abs(e2 - expected)))
np.testing.assert_allclose(e2, expected, atol=1e-5)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(fold.k[:, 0], e2 - Ef, color="tab:orange", lw=2)
plt.plot(fold.k[:, 0], e0 - Ef, "k--", lw=0.7)
plt.plot(fold.k[:, 0], e1 - Ef, "b:", lw=0.7)
plt.xlabel("q₁ in original reciprocal coordinates")
plt.ylabel("E − EF [eV]")
plt.title("Supercell = original q branch + shifted q+½ branch")
plt.ylim(-15, 15)
plt.xlim(-0.25, 0.25)

## 7. Density matrix → real-space density

Required files: **`BN.DM`, geometry, `B.ion.xml`, and `N.ion.xml`**. `DM` retains the reference-cell/neighbor-cell orbital indexing introduced above. `Grid` samples positions in real space. We use the original four-atom `DM` here, so the expected electron count is 16. The many neighbor-cell columns do not multiply that electron count: they supply the periodic orbital contributions needed to reconstruct density in the chosen grid cell.

$$n(\mathbf r)=\sum_{ij\mathbf R}D_{i,j\mathbf R}\phi_i(\mathbf r)\phi_{j\mathbf R}(\mathbf r).$$

`DM.density(grid)` performs this projection. The nonmagnetic DM already includes spin occupations, so **do not multiply by 2**. The result is a positive electron number density in e/Å³; the electrical charge density is $-e n$. Each B contributes 3 valence electrons and each N contributes 5, giving 16 electrons in this cell.

The floating-point argument in `Grid(0.25, ...)` specifies a target spacing in Å. The actual spacing may differ slightly because the grid dimensions must be integers. Create a fresh `Grid` for each calculation, since density is **added to the existing grid**.

In [ ]:
rho = si.Grid(0.25, geometry=DM.geometry)
DM.density(rho)
print("grid shape:", rho.shape)
print("integrated electrons:", rho.grid.sum() * rho.dvolume)
rho.write("BN-density.cube")

### Evaluate the same DM on several grids

Here we evaluate the same DM and PAOs on different real-space grids; the SCF `MeshCutoff` stays fixed. Check whether the integrated density approaches 16 electrons as the grid becomes finer. Convergence need not be monotonic.

In [ ]:
grids = []
for spacing in [0.40, 0.25, 0.15]:
    grid = si.Grid(spacing, geometry=DM.geometry)
    DM.density(grid)
    grids.append(grid)
    electrons = grid.grid.sum() * grid.dvolume
    print(f"{spacing:.2f} Å : {grid.shape}, N = {electrons:.6f}, ΔN = {electrons - 16:+.6f}")

### A 2D slice through a BN layer

The in-plane lattice vectors are oblique. The plot axes show **fractional coordinates $f_1,f_2$**, rather than Cartesian x/y. For each grid, plot the slice nearest the first BN layer ($f_3=1/4$), using the same color scale.

In [ ]:
vmax = max(g.grid.max() for g in grids)
fig, axes = plt.subplots(1, 3, figsize=(12, 3), constrained_layout=True)
for ax, grid in zip(axes, grids):
    iz = round(0.25 * grid.shape[2])
    im = ax.imshow(grid.grid[:, :, iz].T, origin="lower", extent=[0, 1, 0, 1],
                   vmin=0, vmax=vmax, aspect="auto")
    ax.set(title=f"{grid.shape}, f₃={iz / grid.shape[2]:.3f}", xlabel="f₁", ylabel="f₂")
fig.colorbar(im, ax=axes, label="electrons / Å³")

### Compare with SIESTA RHO on the same grid

Required files: `BN.RHO` (`SaveRho true`), the DM, and numerical orbitals. The sisl reader converts SIESTA's stored units. `Grid(reference.shape, lattice=reference.lattice)` creates a fresh array with the same shape, cell, and origin, allowing a pointwise comparison. Inspect both the integrated electron counts and the pointwise RMS difference. PAO interpolation and numerical integration can introduce small differences.

In [ ]:
reference = si.get_sile("BN.RHO").read_grid()
reconstructed = si.Grid(reference.shape, lattice=reference.lattice)
DM.density(reconstructed)
difference = reconstructed.grid - reference.grid
print("SIESTA N:", reference.grid.sum() * reference.dvolume)
print("sisl N:", reconstructed.grid.sum() * reconstructed.dvolume)
print("RMS [e/Å³]:", np.sqrt(np.mean(difference**2)))
print("max |difference| [e/Å³]:", np.max(np.abs(difference)))

## 8. Another core feature: Brillouin-zone averaged DOS

Required files: `BN.HSX` and geometry. `MonkhorstPack` and `apply.average.eigenstate(wrap=...)` calculate a weighted average over a k mesh. This DOS is **per spin channel** for the nonmagnetic Hamiltonian. Multiply by 2 to include both spins. Convergence with respect to the k mesh and broadening should be checked separately.

In [ ]:
bz = si.MonkhorstPack(H, [6, 6, 2])
energy = np.linspace(-15, 15, 301)
dos = bz.apply.average.eigenstate(wrap=lambda state: state.DOS(energy + Ef))
plt.figure(figsize=(6, 3))
plt.plot(energy, dos)
plt.xlabel("E − EF [eV]")
plt.ylabel("DOS [states/eV/cell/spin]")

## 9. Another core feature: eigenstate → wavefunction

Required files: **HSX and `ion.xml`**, with geometry. `H.eigenstate()` returns eigenvalues and eigenvectors together; `wavefunction()` projects a state onto a real-space grid. A nonmagnetic insulator with 16 electrons has 8 occupied bands, so we select index 7 at Γ (using zero-based indexing). This is the highest occupied state at Γ, which need not be the valence-band maximum across the entire BZ.

At Γ, we can use a real-valued grid. A general k point requires `dtype=complex`. For a single normalized state, $\int |\psi|^2 dV$ should be approximately 1; this is distinct from the total electron density.

In [ ]:
state = H.eigenstate(k=[0, 0, 0]).sub(7)
psi = si.Grid(0.15, geometry=H.geometry)
state.wavefunction(psi)
probability = np.abs(psi.grid)**2
print("state energy − EF [eV]:", state.eig - Ef)
print("∫ |ψ|² dV:", probability.sum() * psi.dvolume)
psi.write("BN-Gamma-state7.cube")

## Try it yourself

- Change to `H.tile(3, axis=0)`. How do the reciprocal-coordinate transformation and the number of folded branches change?
- Call `H.eigh(k=...)` away from Γ and check that `Hk` and `Sk` are Hermitian.
- How do the electron-count error and runtime change as you refine the density grid?
- Use `state.PDOS(energy + Ef)` to obtain orbital-resolved DOS. Comparing with the total DOS requires all eigenstates and a k-point average, rather than a single state.

### References

- [sisl SIESTA I/O: FDF reader](https://zerothi.github.io/sisl/api/io/generated/sisl.io.siesta.fdfSileSiesta.html)
- [Hamiltonian: Hk, Sk, eigh, tile](https://zerothi.github.io/sisl/api/generated/sisl.physics.Hamiltonian.html)
- [BandStructure](https://zerothi.github.io/sisl/api/generated/sisl.physics.BandStructure.html)
- [DensityMatrix: density](https://zerothi.github.io/sisl/api/generated/sisl.physics.DensityMatrix.html)
- [Official sisl SIESTA tutorial](https://zerothi.github.io/sisl/tutorials/tutorial_siesta_1.html)
- [SIESTA input/output options](https://docs.siesta-project.org/projects/siesta/en/stable/reference/siesta.html)

## Input provenance and validation

Only the **structure and two PSF files** were copied unchanged from this source directory:

```text
/home2/rong/01.Research/1.Material/5.Bulk/BN-bulk/01.LDA/AA_dot/02.volume_eos/optimized_structure/input
```

The original calculation results, basis settings, and k mesh were not used. `src/inputs/RUN.fdf` was written for this exercise: fixed geometry, LDA/CA, SZP, 150 Ry, and a 6×6×2 k mesh. These are teaching settings, not converged research settings. The PSF headers identify ATM4.2.7, Troullier–Martins, and CA.

| File | SHA-256 |
|---|---|
| `STRUCT.fdf` | `b8b39aea1b53a1f94d03119512f4ae6fe44c7c548e145f80e8c88fe56d98d1b0` |
| `B.psf` | `39905f13d8b36f49bec309d322124dc0de4b3195e9e7ffab2f06fa4b442f9b53` |
| `N.psf` | `d50f574d7497dd8f52e017ea6b9dd34bbe9f66382688f4b9ce4c87d8342799b6` |

### Local validation

Validated locally on 2026-11-03 with SIESTA 5.4.2 nompi, Python 3.12, and sisl 0.16.4: a fresh SCF calculation converged, and all analysis cells ran successfully. The maximum folding error was `1.88e-7 eV`; the finest density grid integrated to `16.000005` electrons; the RMS difference from SIESTA RHO was `2.92e-6 e/Å³`; and the single-state normalization was `1.000004`.

The Colab UI and remote input download have not been tested. The Colab link becomes usable after these files are pushed to `main`. Generated matrices, densities, and Cube files are saved to `/content/bn-matrices`.